                  PDF
                   │
                   ▼
        ┌─────────────────────┐
        │ PyMuPDF              │
        │ Text + Coordinates   │
        └──────────┬──────────┘
                   │
                   ▼
          Section Detection
                   │
       ┌───────────┼───────────┐
       ▼           ▼           ▼
    Contact      Sections     Layout
       │           │
       │     ┌─────┼─────┬────────┐
       │     ▼     ▼     ▼        ▼
       │   Skills Exp  Projects Education
       │     │     │      │        │
       │     └─────┴──────┴────────┘
       │               │
       ▼               ▼
  Regex extraction   LLM extraction
       │               │
       └───────┬───────┘
               ▼
             MERGE
               │
               ▼
        Remove empty fields
               │
               ▼
           VALIDATION
               │
               ▼
          FINAL JSON
          

In [ ]:
!pip install -q streamlit pyngrok

In [ ]:
!pip install -q pymupdf transformers sentence-transformers faiss-cpu bitsandbytes pydantic

In [ ]:
import re
import json
import torch
import pymupdf

from transformers import AutoTokenizer, AutoModelForCausalLM
from pydantic import BaseModel, Field
from typing import Optional, List, Dict, Any

In [ ]:
print("Loading LLM...")

llm_model_id = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(llm_model_id)

llm_model = AutoModelForCausalLM.from_pretrained(
    llm_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("✅ LLM loaded")

In [ ]:
from google.colab import files

uploaded = files.upload()

filename = next(iter(uploaded))

print(f"📄 Uploaded: {filename}")

In [ ]:
doc = pymupdf.open(filename)

layout_blocks = []

for page_num, page in enumerate(doc):

    blocks = page.get_text("blocks")

    for block in blocks:

        x0, y0, x1, y1, text = block[:5]

        text = text.strip()

        if not text:
            continue

        layout_blocks.append({
            "page": page_num + 1,
            "x0": round(x0, 2),
            "y0": round(y0, 2),
            "x1": round(x1, 2),
            "y1": round(y1, 2),
            "text": text
        })

doc.close()

print(f"✅ Extracted {len(layout_blocks)} layout blocks")

In [ ]:
for block in layout_blocks[:30]:

    print(
        f"[Page {block['page']}] "
        f"({block['x0']}, {block['y0']}) → "
        f"{block['text'][:150]}"
    )

In [ ]:
resume_lines = []

for block in layout_blocks:

    lines = block["text"].splitlines()

    for line in lines:

        line = line.strip()

        if len(line) < 2:
            continue

        resume_lines.append({
            "page": block["page"],
            "x0": block["x0"],
            "y0": block["y0"],
            "text": line
        })

print(f"✅ Total lines: {len(resume_lines)}")

In [ ]:
raw_text = "\n".join(
    line["text"]
    for line in resume_lines
)

print(raw_text[:5000])

In [ ]:
SECTION_ALIASES = {

    "summary": {
        "summary",
        "professional summary",
        "profile",
        "about me",
        "objective",
        "career objective"
    },

    "experience": {
        "experience",
        "work experience",
        "professional experience",
        "employment",
        "work history",
        "career history"
    },

    "education": {
        "education",
        "academic background",
        "academic qualifications",
        "qualifications"
    },

    "skills": {
        "skills",
        "technical skills",
        "core skills",
        "technical competencies",
        "competencies",
        "technologies",
        "tools",
        "expertise"
    },

    "projects": {
        "projects",
        "personal projects",
        "academic projects",
        "key projects",
        "selected projects"
    },

    "certifications": {
        "certifications",
        "certificates",
        "licenses",
        "professional certifications"
    },

    "achievements": {
        "achievements",
        "awards",
        "honors",
        "accomplishments"
    },

    "languages": {
        "languages",
        "language proficiency"
    }
}

In [ ]:
def normalize_heading(text):

    text = text.lower().strip()

    # Remove decorative characters
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [ ]:
def detect_section_heading(text):

    normalized = normalize_heading(text)

    for section, aliases in SECTION_ALIASES.items():

        if normalized in aliases:
            return section

    return None

In [ ]:
def split_into_sections(lines):

    sections = {
        "header": []
    }

    current_section = "header"

    for line in lines:

        text = line["text"]

        detected = detect_section_heading(text)

        if detected:

            current_section = detected

            if current_section not in sections:
                sections[current_section] = []

            continue

        sections[current_section].append(line)

    return sections

In [ ]:
sections = split_into_sections(resume_lines)

print("Detected sections:")

for section, lines in sections.items():

    print(
        f"  {section}: {len(lines)} lines"
    )

In [ ]:
for section, lines in sections.items():

    print("\n" + "=" * 70)
    print(section.upper())
    print("=" * 70)

    text = "\n".join(
        line["text"]
        for line in lines
    )

    print(text[:3000])

In [ ]:
def section_text(sections, section_name):

    if section_name not in sections:
        return ""

    return "\n".join(
        line["text"]
        for line in sections[section_name]
    ).strip()

In [ ]:
def run_llm(prompt, max_new_tokens=500):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a precise resume information extraction system. "
                "Extract only information explicitly present in the input. "
                "Never guess or hallucinate."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(llm_model.device)

    with torch.no_grad():

        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05
        )

    generated_tokens = outputs[
        0
    ][
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [ ]:
def parse_json_response(response):

    response = response.strip()

    # Remove markdown code fences
    response = re.sub(
        r"```json",
        "",
        response,
        flags=re.IGNORECASE
    )

    response = response.replace("```", "").strip()

    # Find JSON object
    start = response.find("{")
    end = response.rfind("}")

    if start == -1 or end == -1:
        return {}

    json_text = response[start:end + 1]

    try:
        return json.loads(json_text)

    except json.JSONDecodeError:

        print("⚠️ Invalid JSON returned by model")
        print(response)

        return {}

In [ ]:
def extract_emails(text):

    pattern = (
        r'\b[A-Za-z0-9._%+-]+'
        r'@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b'
    )

    emails = re.findall(pattern, text)

    return list(dict.fromkeys(emails))

In [ ]:
def extract_urls(text):

    pattern = (
        r'https?://[^\s<>"\']+'
        r'|(?:www\.)?[A-Za-z0-9.-]+\.[A-Za-z]{2,}'
        r'(?:/[^\s<>"\']*)?'
    )

    urls = re.findall(pattern, text)

    cleaned = []

    for url in urls:

        url = url.rstrip(".,;)")

        if url not in cleaned:
            cleaned.append(url)

    return cleaned

In [ ]:
def classify_urls(urls):

    result = {}

    for url in urls:

        lower = url.lower()

        if "linkedin.com" in lower:

            result.setdefault(
                "linkedin",
                []
            ).append(url)

        elif "github.com" in lower:

            result.setdefault(
                "github",
                []
            ).append(url)

        elif any(
            keyword in lower
            for keyword in [
                "portfolio",
                "personal",
                "website"
            ]
        ):

            result.setdefault(
                "portfolio",
                []
            ).append(url)

        else:

            result.setdefault(
                "other",
                []
            ).append(url)

    return result

In [ ]:
def extract_phones(text):

    pattern = (
        r'(?<!\d)'
        r'(?:\+?\d{1,3}[\s.-]?)?'
        r'(?:\(?\d{3}\)?[\s.-]?)?'
        r'\d{3}[\s.-]\d{4}'
        r'(?!\d)'
    )

    phones = re.findall(pattern, text)

    phones = [
        re.sub(r'\s+', ' ', phone).strip()
        for phone in phones
    ]

    return list(dict.fromkeys(phones))

In [ ]:
header_text = section_text(
    sections,
    "header"
)

name_prompt = f"""
Extract the person's full name from the following resume header.

Rules:
- Return only the name.
- Do not invent a name.
- Do not return email, phone, job title, or company.
- If no clear name exists, return an empty string.

HEADER:

{header_text[:2000]}

Return JSON:

{{
  "name": "..."
}}
"""

name_response = run_llm(
    name_prompt,
    max_new_tokens=100
)

name_data = parse_json_response(
    name_response
)

print(name_data)

In [ ]:
def extract_skills(text):

    if not text.strip():
        return {}

    prompt = f"""
Extract ONLY explicit skills from this SKILLS section.

Allowed:
- programming languages
- frameworks
- libraries
- databases
- cloud platforms
- developer tools
- software tools
- methodologies
- technical competencies

DO NOT extract:
- project names
- project descriptions
- job responsibilities
- achievements
- action phrases
- sentences
- company names
- job titles
- technologies merely inferred from a description

Only extract skills explicitly written in the section.

Return JSON only:

{{
    "skills": []
}}

SKILLS SECTION:

{text}
"""

    response = run_llm(
        prompt,
        max_new_tokens=400
    )

    return parse_json_response(response)

In [ ]:
skills_data = extract_skills(
    section_text(sections, "skills")
)

print(
    json.dumps(
        skills_data,
        indent=2
    )
)

In [ ]:
def extract_experience(text):

    if not text.strip():
        return {}

    prompt = f"""
Extract ONLY work experience from the following EXPERIENCE section.

For every distinct job role create one object.

Fields:

company
role
location
start_date
end_date
description

Rules:

- Use only information explicitly present.
- Do not infer company names.
- Do not infer job titles.
- Do not invent dates.
- Preserve dates as written.
- If the resume says Present, use "Present".
- Keep descriptions faithful to the resume.
- Do not move projects into experience.
- Do not move skills into experience.

Omit fields that are not explicitly present.

Return JSON only:

{{
    "experience": [
        {{
            "company": "...",
            "role": "...",
            "location": "...",
            "start_date": "...",
            "end_date": "...",
            "description": "..."
        }}
    ]
}}

EXPERIENCE SECTION:

{text}
"""

    response = run_llm(
        prompt,
        max_new_tokens=700
    )

    return parse_json_response(response)

In [ ]:
experience_data = extract_experience(
    section_text(sections, "experience")
)

print(
    json.dumps(
        experience_data,
        indent=2
    )
)

In [ ]:
def extract_projects(text):

    if not text.strip():
        return {}

    prompt = f"""
Extract ONLY projects from this PROJECTS section.

For each project extract:

name
description
technologies
links

Rules:

- Project names must come from the text.
- Descriptions must come from the text.
- Technologies must be explicitly mentioned.
- Never infer technologies.
- Never turn an action phrase into a technology.
- Never turn a sentence into a technology.
- Never extract job responsibilities.
- Never invent URLs.
- Omit fields that are not present.

Return JSON only:

{{
    "projects": [
        {{
            "name": "...",
            "description": "...",
            "technologies": [],
            "links": []
        }}
    ]
}}

PROJECTS SECTION:

{text}
"""

    response = run_llm(
        prompt,
        max_new_tokens=700
    )

    return parse_json_response(response)

In [ ]:
projects_data = extract_projects(
    section_text(sections, "projects")
)

print(
    json.dumps(
        projects_data,
        indent=2
    )
)

In [ ]:
def extract_education(text):

    if not text.strip():
        return {}

    prompt = f"""
Extract ONLY education information from this EDUCATION section.

For each education entry extract:

institution
degree
field_of_study
location
start_date
end_date

Rules:

- Only use explicit information.
- Do not infer degree names.
- Do not infer field of study.
- Do not invent dates.
- Omit missing fields.

Return JSON only:

{{
    "education": [
        {{
            "institution": "...",
            "degree": "...",
            "field_of_study": "...",
            "location": "...",
            "start_date": "...",
            "end_date": "..."
        }}
    ]
}}

EDUCATION SECTION:

{text}
"""

    response = run_llm(
        prompt,
        max_new_tokens=500
    )

    return parse_json_response(response)

In [ ]:
education_data = extract_education(
    section_text(sections, "education")
)

print(
    json.dumps(
        education_data,
        indent=2
    )
)

In [ ]:
def extract_summary(text):

    if not text.strip():
        return {}

    prompt = f"""
Extract the professional summary from this SUMMARY section.

Rules:

- Preserve the meaning of the original text.
- Do not add information.
- Do not rewrite it into new claims.
- Do not extract skills separately.
- If there is no summary, return {{}}.

Return JSON only:

{{
    "summary": "..."
}}

SUMMARY SECTION:

{text}
"""

    response = run_llm(
        prompt,
        max_new_tokens=300
    )

    return parse_json_response(response)

In [ ]:
summary_data = extract_summary(
    section_text(sections, "summary")
)

print(
    json.dumps(
        summary_data,
        indent=2
    )
)

In [ ]:
def extract_certifications(text):

    if not text.strip():
        return {}

    prompt = f"""
Extract certifications from this CERTIFICATIONS section.

For each certification extract:

name
issuer
date

Only use information explicitly present.

Omit missing fields.

Return JSON only:

{{
    "certifications": [
        {{
            "name": "...",
            "issuer": "...",
            "date": "..."
        }}
    ]
}}

CERTIFICATIONS:

{text}
"""

    response = run_llm(
        prompt,
        max_new_tokens=400
    )

    return parse_json_response(response)

In [ ]:
certifications_data = extract_certifications(
    section_text(sections, "certifications")
)

print(
    json.dumps(
        certifications_data,
        indent=2
    )
)

In [ ]:
def extract_languages(text):

    if not text.strip():
        return {}

    prompt = f"""
Extract spoken/human languages from this LANGUAGES section.

Important:

- Extract human languages only.
- Do NOT extract Python, Java, JavaScript, SQL, etc.
- Preserve proficiency information when explicitly provided.

Return JSON only:

{{
    "languages": []
}}

LANGUAGES:

{text}
"""

    response = run_llm(
        prompt,
        max_new_tokens=300
    )

    return parse_json_response(response)

In [ ]:
languages_data = extract_languages(
    section_text(sections, "languages")
)

print(
    json.dumps(
        languages_data,
        indent=2
    )
)

In [ ]:
emails = extract_emails(raw_text)

phones = extract_phones(raw_text)

urls = extract_urls(raw_text)

classified_urls = classify_urls(urls)

print("Emails:")
print(emails)

print("\nPhones:")
print(phones)

print("\nURLs:")
print(urls)

print("\nClassified URLs:")
print(
    json.dumps(
        classified_urls,
        indent=2
    )
)

In [ ]:
final_data = {}

# Name
if name_data.get("name"):
    final_data["name"] = name_data["name"]

# Contact
if emails:
    final_data["email"] = emails[0]

if phones:
    final_data["phone"] = phones[0]

# Links
if classified_urls:
    final_data["links"] = classified_urls

# Semantic extraction
final_data.update(summary_data)
final_data.update(skills_data)
final_data.update(experience_data)
final_data.update(education_data)
final_data.update(projects_data)
final_data.update(certifications_data)
final_data.update(languages_data)

print(
    json.dumps(
        final_data,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
def remove_empty_values(obj):

    if isinstance(obj, dict):

        cleaned = {}

        for key, value in obj.items():

            value = remove_empty_values(value)

            if value in [
                None,
                "",
                [],
                {}
            ]:
                continue

            cleaned[key] = value

        return cleaned

    elif isinstance(obj, list):

        cleaned_list = []

        for item in obj:

            item = remove_empty_values(item)

            if item not in [
                None,
                "",
                [],
                {}
            ]:
                cleaned_list.append(item)

        return cleaned_list

    return obj

In [ ]:
final_data = remove_empty_values(
    final_data
)

print(
    json.dumps(
        final_data,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
class Experience(BaseModel):

    company: Optional[str] = None
    role: Optional[str] = None
    location: Optional[str] = None
    start_date: Optional[str] = None
    end_date: Optional[str] = None
    description: Optional[str] = None


class Education(BaseModel):

    institution: Optional[str] = None
    degree: Optional[str] = None
    field_of_study: Optional[str] = None
    location: Optional[str] = None
    start_date: Optional[str] = None
    end_date: Optional[str] = None


class Project(BaseModel):

    name: Optional[str] = None
    description: Optional[str] = None
    technologies: Optional[List[str]] = None
    links: Optional[List[str]] = None

In [ ]:
def validate_resume(data):

    errors = []

    if "experience" in data:

        if not isinstance(
            data["experience"],
            list
        ):
            errors.append(
                "experience must be a list"
            )

    if "education" in data:

        if not isinstance(
            data["education"],
            list
        ):
            errors.append(
                "education must be a list"
            )

    if "projects" in data:

        if not isinstance(
            data["projects"],
            list
        ):
            errors.append(
                "projects must be a list"
            )

    if "skills" in data:

        if not isinstance(
            data["skills"],
            list
        ):
            errors.append(
                "skills must be a list"
            )

    return errors

In [ ]:
validation_errors = validate_resume(
    final_data
)

if validation_errors:

    print("❌ Validation errors:")

    for error in validation_errors:
        print("-", error)

else:

    print("✅ Resume JSON passed validation")

In [ ]:
print("\n" + "=" * 80)
print("FINAL RESUME JSON")
print("=" * 80)

print(
    json.dumps(
        final_data,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
%%writefile app.py

import os
import re
import json
import tempfile
import torch
import pymupdf
import streamlit as st

from transformers import AutoTokenizer, AutoModelForCausalLM


# =========================================================
# PAGE CONFIG
# =========================================================

st.set_page_config(
    page_title="Resume Dossier & Parsing Engine",
    page_icon="📋",
    layout="wide"
)


# =========================================================
# CUSTOM CSS (CLEAN SAAS / DOSSIER THEME)
# =========================================================

st.markdown("""
<style>
/* Base typography and layout reset */
.main .block-container {
    padding-top: 2rem;
    padding-bottom: 4rem;
    max-width: 1120px;
}
h1, h2, h3, h4, h5 {
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
    letter-spacing: -0.02em;
    color: #0f172a;
}

/* Candidate Dossier Header */
.profile-card {
    background: #ffffff;
    border: 1px solid #e2e8f0;
    border-radius: 12px;
    padding: 24px;
    margin-bottom: 24px;
    box-shadow: 0 1px 3px rgba(0, 0, 0, 0.04);
}
.profile-name {
    font-size: 26px;
    font-weight: 700;
    color: #0f172a;
    margin: 0;
}
.profile-meta {
    font-size: 14px;
    color: #475569;
    margin-top: 10px;
    display: flex;
    flex-wrap: wrap;
    gap: 16px;
    align-items: center;
}
.profile-meta a {
    color: #2563eb;
    text-decoration: none;
    font-weight: 500;
}
.profile-meta a:hover {
    text-decoration: underline;
}

/* Section Containers & Chronology Blocks */
.section-block {
    background: #ffffff;
    border: 1px solid #e2e8f0;
    border-radius: 10px;
    padding: 18px 20px;
    margin-bottom: 14px;
    box-shadow: 0 1px 2px rgba(0, 0, 0, 0.02);
}
.entry-header {
    display: flex;
    justify-content: space-between;
    align-items: baseline;
    margin-bottom: 4px;
}
.entry-title {
    font-weight: 600;
    font-size: 15px;
    color: #0f172a;
}
.entry-subtitle {
    font-size: 13.5px;
    color: #475569;
    margin-bottom: 8px;
    font-weight: 500;
}
.entry-date {
    font-size: 12.5px;
    color: #64748b;
    font-variant-numeric: tabular-nums;
}

/* Metadata Badges */
.badge {
    display: inline-block;
    background: #f1f5f9;
    color: #334155;
    font-size: 12px;
    font-weight: 500;
    padding: 3px 9px;
    border-radius: 6px;
    margin: 2px 4px 4px 0;
    border: 1px solid #e2e8f0;
}
</style>
""", unsafe_allow_html=True)


# =========================================================
# LOAD MODEL
# =========================================================

@st.cache_resource
def load_model():
    model_id = "Qwen/Qwen2.5-3B-Instruct"

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    return tokenizer, model


# =========================================================
# LLM & JSON HELPERS
# =========================================================

def run_llm(tokenizer, model, prompt, max_new_tokens=500):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a precise resume information extraction system. "
                "Extract only information explicitly present in the input. "
                "Never guess or hallucinate."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return response.strip()


def parse_json_response(response):
    response = response.strip()
    response = re.sub(r"```json", "", response, flags=re.IGNORECASE)
    response = response.replace("```", "").strip()

    start = response.find("{")
    end = response.rfind("}")

    if start == -1 or end == -1:
        return {}

    try:
        return json.loads(response[start:end + 1])
    except Exception:
        return {}


# =========================================================
# PDF EXTRACTION & GEOMETRY
# =========================================================

def extract_pdf(pdf_path):
    doc = pymupdf.open(pdf_path)
    layout_blocks = []

    for page_num, page in enumerate(doc):
        blocks = page.get_text("blocks")
        for block in blocks:
            x0, y0, x1, y1, text = block[:5]
            text = text.strip()
            if not text:
                continue

            layout_blocks.append({
                "page": page_num + 1,
                "x0": round(x0, 2),
                "y0": round(y0, 2),
                "x1": round(x1, 2),
                "y1": round(y1, 2),
                "text": text
            })

    doc.close()
    return layout_blocks


def create_lines(layout_blocks):
    lines = []
    for block in layout_blocks:
        for line in block["text"].splitlines():
            line = line.strip()
            if len(line) < 2:
                continue
            lines.append({
                "page": block["page"],
                "x0": block["x0"],
                "y0": block["y0"],
                "text": line
            })
    return lines


# =========================================================
# SECTION DETECTION & SPLITTING
# =========================================================

SECTION_ALIASES = {
    "summary": {
        "summary", "professional summary", "profile", "about me", "objective", "career objective"
    },
    "experience": {
        "experience", "work experience", "professional experience", "employment", "work history", "career history"
    },
    "education": {
        "education", "academic background", "academic qualifications", "qualifications"
    },
    "skills": {
        "skills", "technical skills", "core skills", "technical competencies", "competencies", "technologies", "tools", "expertise"
    },
    "projects": {
        "projects", "personal projects", "academic projects", "key projects", "selected projects"
    },
    "certifications": {
        "certifications", "certificates", "licenses", "professional certifications"
    },
    "achievements": {
        "achievements", "awards", "honors", "accomplishments"
    },
    "languages": {
        "languages", "language proficiency"
    }
}


def normalize_heading(text):
    text = text.lower().strip()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def detect_section_heading(text):
    normalized = normalize_heading(text)
    for section, aliases in SECTION_ALIASES.items():
        if normalized in aliases:
            return section
    return None


def split_into_sections(lines):
    sections = {"header": []}
    current_section = "header"

    for line in lines:
        detected = detect_section_heading(line["text"])
        if detected:
            current_section = detected
            if current_section not in sections:
                sections[current_section] = []
            continue
        sections[current_section].append(line)

    return sections


def section_text(sections, section_name):
    if section_name not in sections:
        return ""
    return "\n".join(line["text"] for line in sections[section_name]).strip()


# =========================================================
# DETERMINISTIC CONTACT PARSERS
# =========================================================

def extract_emails(text):
    pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b'
    return list(dict.fromkeys(re.findall(pattern, text)))


def extract_phones(text):
    pattern = (
        r'(?<!\d)'
        r'(?:\+?\d{1,3}[\s.-]?)?'
        r'(?:\(?\d{3}\)?[\s.-]?)?'
        r'\d{3}[\s.-]\d{4}'
        r'(?!\d)'
    )
    return list(dict.fromkeys(re.findall(pattern, text)))


def extract_urls(text):
    pattern = (
        r'https?://[^\s<>"\']+'
        r'|(?:www\.)?[A-Za-z0-9.-]+\.[A-Za-z]{2,}'
        r'(?:/[^\s<>"\']*)?'
    )
    urls = re.findall(pattern, text)
    cleaned = []
    for url in urls:
        url = url.rstrip(".,;)")
        if url not in cleaned:
            cleaned.append(url)
    return cleaned


def classify_urls(urls):
    result = {}
    for url in urls:
        lower = url.lower()
        if "linkedin.com" in lower:
            result.setdefault("linkedin", []).append(url)
        elif "github.com" in lower:
            result.setdefault("github", []).append(url)
        elif any(k in lower for k in ["portfolio", "personal", "website"]):
            result.setdefault("portfolio", []).append(url)
        else:
            result.setdefault("other", []).append(url)
    return result


# =========================================================
# SPECIALIZED LLM EXTRACTORS
# =========================================================

def extract_name(tokenizer, model, header_text):
    prompt = f"""
Extract the person's full name from the resume header.

Rules:
- Return only the person's name.
- Do not return email, phone, or job title.
- Do not invent a name.
- If no clear name exists, return {{}}.

HEADER:
{header_text[:2000]}

Return JSON only:
{{
    "name": "..."
}}
"""
    response = run_llm(tokenizer, model, prompt, 100)
    return parse_json_response(response)


def extract_skills(tokenizer, model, text):
    if not text.strip():
        return {}

    prompt = f"""
Extract ONLY explicit skills from this SKILLS section.

Allowed:
- programming languages, frameworks, libraries, databases, cloud platforms, developer tools, software tools, methodologies, technical competencies.

DO NOT extract:
- project names, project descriptions, job responsibilities, achievements, action phrases, sentences, company names, job titles.

Return JSON only:
{{
    "skills": []
}}

SKILLS:
{text}
"""
    response = run_llm(tokenizer, model, prompt, 400)
    return parse_json_response(response)


def extract_experience(tokenizer, model, text):
    if not text.strip():
        return {}

    prompt = f"""
Extract ONLY work experience from this EXPERIENCE section.
Create one object for every distinct role.

Fields:
company, role, location, start_date, end_date, description

Rules:
- Only use information explicitly present.
- Do not infer company names, job titles, or dates.
- Preserve dates exactly. If resume says Present, use Present.
- Omit missing fields.

Return JSON only:
{{
    "experience": [
        {{
            "company": "...",
            "role": "...",
            "location": "...",
            "start_date": "...",
            "end_date": "...",
            "description": "..."
        }}
    ]
}}

EXPERIENCE:
{text}
"""
    response = run_llm(tokenizer, model, prompt, 700)
    return parse_json_response(response)


def extract_projects(tokenizer, model, text):
    if not text.strip():
        return {}

    prompt = f"""
Extract ONLY projects from this PROJECTS section.

Fields:
name, description, technologies, links

Rules:
- Project names and descriptions must come directly from text.
- Technologies must be explicitly mentioned (do not infer).
- Never invent URLs. Omit missing fields.

Return JSON only:
{{
    "projects": [
        {{
            "name": "...",
            "description": "...",
            "technologies": [],
            "links": []
        }}
    ]
}}

PROJECTS:
{text}
"""
    response = run_llm(tokenizer, model, prompt, 700)
    return parse_json_response(response)


def extract_education(tokenizer, model, text):
    if not text.strip():
        return {}

    prompt = f"""
Extract ONLY education information.

Fields:
institution, degree, field_of_study, location, start_date, end_date

Rules:
- Only use explicit information.
- Do not infer degrees or fields of study.
- Omit missing fields.

Return JSON only:
{{
    "education": [
        {{
            "institution": "...",
            "degree": "...",
            "field_of_study": "...",
            "location": "...",
            "start_date": "...",
            "end_date": "..."
        }}
    ]
}}

EDUCATION:
{text}
"""
    response = run_llm(tokenizer, model, prompt, 500)
    return parse_json_response(response)


def extract_summary(tokenizer, model, text):
    if not text.strip():
        return {}

    prompt = f"""
Extract the professional summary from this SUMMARY section.

Rules:
- Preserve original meaning without adding claims.
- If no summary exists, return {{}}.

Return JSON only:
{{
    "summary": "..."
}}

SUMMARY:
{text}
"""
    response = run_llm(tokenizer, model, prompt, 300)
    return parse_json_response(response)


def remove_empty_values(obj):
    if isinstance(obj, dict):
        cleaned = {}
        for k, v in obj.items():
            v = remove_empty_values(v)
            if v not in [None, "", [], {}]:
                cleaned[k] = v
        return cleaned
    elif isinstance(obj, list):
        cleaned = []
        for item in obj:
            item = remove_empty_values(item)
            if item not in [None, "", [], {}]:
                cleaned.append(item)
        return cleaned
    return obj


# =========================================================
# ORCHESTRATION PIPELINE
# =========================================================

def parse_resume(pdf_path, tokenizer, model):
    blocks = extract_pdf(pdf_path)
    lines = create_lines(blocks)
    raw_text = "\n".join(line["text"] for line in lines)
    sections = split_into_sections(lines)

    emails = extract_emails(raw_text)
    phones = extract_phones(raw_text)
    urls = extract_urls(raw_text)
    classified_urls = classify_urls(urls)

    name_data = extract_name(tokenizer, model, section_text(sections, "header"))
    skills_data = extract_skills(tokenizer, model, section_text(sections, "skills"))
    experience_data = extract_experience(tokenizer, model, section_text(sections, "experience"))
    projects_data = extract_projects(tokenizer, model, section_text(sections, "projects"))
    education_data = extract_education(tokenizer, model, section_text(sections, "education"))
    summary_data = extract_summary(tokenizer, model, section_text(sections, "summary"))

    result = {}
    if name_data.get("name"):
        result["name"] = name_data["name"]
    if emails:
        result["email"] = emails[0]
    if phones:
        result["phone"] = phones[0]
    if classified_urls:
        result["links"] = classified_urls

    result.update(summary_data)
    result.update(skills_data)
    result.update(experience_data)
    result.update(projects_data)
    result.update(education_data)

    return remove_empty_values(result), sections


# =========================================================
# UI - SIDEBAR INGESTION CONTROLS
# =========================================================

with st.sidebar:
    st.markdown("### Ingestion Controls")
    st.caption("Load a candidate PDF resume to extract structured schema fields.")

    uploaded_file = st.file_uploader(
        "Resume Document",
        type=["pdf"],
        label_visibility="collapsed"
    )

    parse_trigger = st.button(
        "Extract Schema",
        type="primary",
        disabled=not uploaded_file,
        use_container_width=True
    )

    if uploaded_file:
        st.divider()
        st.markdown(f"**Loaded:** `{uploaded_file.name}`")
        st.caption(f"Size: {round(uploaded_file.size / 1024, 1)} KB")


# =========================================================
# UI - WORKSPACE & DOSSIER VIEW
# =========================================================

if "parsed_data" not in st.session_state:
    st.session_state.parsed_data = None
    st.session_state.sections = None

if parse_trigger and uploaded_file:
    tokenizer, model = load_model()

    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
        tmp.write(uploaded_file.getbuffer())
        pdf_path = tmp.name

    try:
        with st.status("Analyzing document structure...", expanded=False) as status:
            st.write("Extracting layout geometry and text blocks...")
            st.write("Resolving boundaries and semantic sections...")
            result, sections = parse_resume(pdf_path, tokenizer, model)
            status.update(label="Extraction completed", state="complete")

        st.session_state.parsed_data = result
        st.session_state.sections = sections

    except Exception as err:
        st.error(f"Processing error encountered: {str(err)}")
    finally:
        if os.path.exists(pdf_path):
            os.remove(pdf_path)


# Render dossier when state holds data
if st.session_state.parsed_data:
    data = st.session_state.parsed_data
    sections = st.session_state.sections

    # 1. Candidate Dossier Header
    contact_parts = []
    if data.get("email"):
        contact_parts.append(f"<span>✉️ {data['email']}</span>")
    if data.get("phone"):
        contact_parts.append(f"<span>📞 {data['phone']}</span>")

    if data.get("links"):
        for category, urls in data["links"].items():
            for url in urls:
                label = category.capitalize()
                contact_parts.append(f'<a href="{url}" target="_blank">{label}</a>')

    meta_html = "".join(contact_parts)
    candidate_name = data.get("name", "Candidate Profile")

    st.markdown(f"""
    <div class="profile-card">
        <h2 class="profile-name">{candidate_name}</h2>
        <div class="profile-meta">{meta_html}</div>
    </div>
    """, unsafe_allow_html=True)

    # 2. Tabs
    tab_overview, tab_history, tab_debug = st.tabs([
        "Profile & Core Competencies",
        "Chronology & Projects",
        "Normalized Schema & Debug"
    ])

    with tab_overview:
        col_summary, col_skills = st.columns([1.2, 1], gap="medium")

        with col_summary:
            st.markdown("#### Professional Background")
            if data.get("summary"):
                st.write(data["summary"])
            else:
                st.caption("No explicit summary block detected.")

            st.markdown("---")
            st.markdown("#### Education")
            if data.get("education"):
                for edu in data["education"]:
                    inst = edu.get("institution", "Institution")
                    deg = edu.get("degree", "")
                    field = f" in {edu['field_of_study']}" if edu.get("field_of_study") else ""
                    dates = f"{edu.get('start_date', '')} – {edu.get('end_date', '')}".strip(" –")

                    st.markdown(f"""
                    <div style="margin-bottom: 14px;">
                        <div style="font-weight: 600; color: #0f172a;">{inst}</div>
                        <div style="font-size: 13.5px; color: #475569;">{deg}{field}</div>
                        <div style="font-size: 12px; color: #94a3b8;">{dates}</div>
                    </div>
                    """, unsafe_allow_html=True)
            else:
                st.caption("No explicit education records detected.")

        with col_skills:
            st.markdown("#### Competencies")
            if data.get("skills"):
                badges = "".join([f'<span class="badge">{s}</span>' for s in data["skills"]])
                st.markdown(f'<div style="line-height: 1.8;">{badges}</div>', unsafe_allow_html=True)
            else:
                st.caption("No explicit skills parsed.")

    with tab_history:
        col_work, col_proj = st.columns(2, gap="large")

        with col_work:
            st.markdown("#### Experience")
            if data.get("experience"):
                for exp in data["experience"]:
                    role = exp.get("role", "Role")
                    company = exp.get("company", "")
                    loc = f" • {exp['location']}" if exp.get("location") else ""
                    dates = f"{exp.get('start_date', '')} – {exp.get('end_date', '')}".strip(" –")
                    desc = exp.get("description", "")

                    st.markdown(f"""
                    <div class="section-block">
                        <div class="entry-header">
                            <span class="entry-title">{role}</span>
                            <span class="entry-date">{dates}</span>
                        </div>
                        <div class="entry-subtitle">{company}{loc}</div>
                        <p style="font-size: 13px; color: #334155; margin: 0; line-height: 1.5;">{desc}</p>
                    </div>
                    """, unsafe_allow_html=True)
            else:
                st.caption("No explicit roles recorded.")

        with col_proj:
            st.markdown("#### Projects")
            if data.get("projects"):
                for proj in data["projects"]:
                    name = proj.get("name", "Project")
                    desc = proj.get("description", "")
                    techs = "".join([f'<span class="badge">{t}</span>' for t in proj.get("technologies", [])])

                    st.markdown(f"""
                    <div class="section-block">
                        <div class="entry-title">{name}</div>
                        <p style="font-size: 13px; color: #334155; margin: 6px 0 10px 0; line-height: 1.5;">{desc}</p>
                        <div>{techs}</div>
                    </div>
                    """, unsafe_allow_html=True)
            else:
                st.caption("No standalone projects parsed.")

    with tab_debug:
        col_json, col_raw = st.columns(2, gap="medium")

        with col_json:
            st.markdown("#### Normalized JSON")
            json_blob = json.dumps(data, indent=2, ensure_ascii=False)
            st.download_button(
                "Download JSON",
                data=json_blob,
                file_name=(
                    uploaded_file.name.replace(".pdf", "") + "_parsed.json"
                    if uploaded_file else "resume_parsed.json"
                ),
                mime="application/json"
            )
            st.code(json_blob, language="json")

        with col_raw:
            st.markdown("#### Block Segmentation Inspection")
            if sections:
                selected_sec = st.selectbox("Section Boundary", list(sections.keys()))
                sec_content = "\n".join(x["text"] for x in sections[selected_sec])
                st.text_area("Segment Lines", value=sec_content, height=450, disabled=True)

elif not uploaded_file:
    st.info("Upload a candidate PDF via the sidebar to generate a dossier.")

In [ ]:
!streamlit run app.py &>/content/streamlit.log &

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("TOKEN")

public_url = ngrok.connect(8501)

print("🚀 Streamlit App:")
print(public_url)